# Calculation Memory — Payoff Matrix
## Planting Density Strategy — Soybeans & Corn

**Data source:** `harvest_summary_brazil.csv`  
**Goal:** Extract real productivity values (sacas/ha) and calculate all Payoff Matrix criteria (Maximax, Maximin, Laplace, and Savage) for both crops.

> **Unit:** productivity is expressed in **sacas/ha** (1 saca = 60 kg — Brazilian standard for soybeans and corn). The raw CSV stores `average_yield` in kg/ha and is converted on load.

---

## 1. Imports and Data Loading

In [1]:
import pandas as pd
import numpy as np

KG_PER_SACA = 60  # 1 saca (sack) = 60 kg — Brazilian standard for soybeans and corn

df = pd.read_csv('harvest_summary_brazil.csv')

# Convert productivity from kg/ha to sacas/ha (1 saca = 60 kg)
df['average_yield'] = df['average_yield'] / KG_PER_SACA

print(f'Total records: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print('Note: average_yield converted to sacas/ha (kg/ha / 60)')
df.head()

Total records: 89,902
Columns: ['climate_account_id', 'field_uuid', 'field_name', 'concrete_growing_season_name', 'crop_name', 'productivity_uom', 'average_yield', 'total_production', 'grain_moisture']
Note: average_yield converted to sacas/ha (kg/ha / 60)


,climate_account_id,field_uuid,field_name,concrete_growing_season_name,crop_name,productivity_uom,average_yield,total_production,grain_moisture
0,247710,62ed29af-baa3-42de-b9ec-8a889df0e42c,Rancho Areião,2026:corn:BR:2:SAF,corn,NaN,95.033333,88148.61,18.7
1,841436,71c2cab0-5997-4e92-bd2a-da2ebe4e9aa6,Mato Grande - 12,2025:soybeans:BR:1:SUM,soybeans,NaN,86.550000,536163.99,17.8
2,649071,4dd4fd1c-cac3-4c62-bbd5-7bc03847f547,Sirlei/ Lebón Régis,2026:beans:BR:2:SAF,beans,NaN,74.783333,559276.50,18.4
3,625510,4876b6f5-4b4c-4026-a037-eaceea48ab06,Agr 020,2025:soybeans:BR:1:SUM,soybeans,NaN,96.666667,307241.70,17.4
4,615581,015db632-e5c0-4aa7-817e-a0dc7c19e888,Areia,2026:corn:BR:2:SAF,corn,NaN,78.483333,507409.87,15.0


---
# PART 1 — SOYBEANS

## 2. Filter — Soybeans Only

In [2]:
soy = df[df['crop_name'] == 'soybeans'].copy()

print(f'Soybean records: {len(soy):,}')
print(f'Seasons: {sorted(soy["concrete_growing_season_name"].unique())}')
print(f'\nYield statistics (average_yield, sacas/ha):')
soy['average_yield'].describe().round(1)

Soybean records: 50,012
Seasons: ['2025:soybeans:BR:1:SUM', '2025:soybeans:BR:2:SAF', '2026:soybeans:BR:1:SUM', '2026:soybeans:BR:2:SAF']

Yield statistics (average_yield, sacas/ha):


count    50012.0
mean        84.9
std         10.6
min         66.7
25%         75.6
50%         84.9
75%         94.0
max        103.3
Name: average_yield, dtype: float64

## 3. Scenario Definition via Terciles

Since the CSV does not have a climate column, scenarios are derived from the **real yield distribution split into terciles**:

- **Pessimistic** → bottom tercile (lowest yields = adverse conditions)
- **Baseline** → middle tercile (normal conditions)
- **Optimistic** → top tercile (highest yields = favourable conditions)

In [3]:
t33_soy = soy['average_yield'].quantile(0.33)
t66_soy = soy['average_yield'].quantile(0.66)

print(f'P33 cut (pessimistic / baseline): {t33_soy:.1f} sc/ha')
print(f'P66 cut (baseline / optimistic):  {t66_soy:.1f} sc/ha')

soy_pess = soy[soy['average_yield'] <= t33_soy]['average_yield']
soy_base = soy[(soy['average_yield'] > t33_soy) & (soy['average_yield'] <= t66_soy)]['average_yield']
soy_opt  = soy[soy['average_yield'] > t66_soy]['average_yield']

print(f'\nPessimistic: n={len(soy_pess):,}  range={soy_pess.min():.1f}–{soy_pess.max():.1f} sc/ha')
print(f'Baseline:    n={len(soy_base):,}  range={soy_base.min():.1f}–{soy_base.max():.1f} sc/ha')
print(f'Optimistic:  n={len(soy_opt):,}  range={soy_opt.min():.1f}–{soy_opt.max():.1f} sc/ha')

P33 cut (pessimistic / baseline): 78.6 sc/ha
P66 cut (baseline / optimistic):  90.8 sc/ha

Pessimistic: n=16,515  range=66.7–78.6 sc/ha
Baseline:    n=16,498  range=78.6–90.8 sc/ha
Optimistic:  n=16,999  range=90.8–103.3 sc/ha


## 4. Real Yield Means per Scenario (Standard Density)

The mean yield of each tercile represents **Standard Density** in the Payoff Matrix — these are the values directly observed in the data.

In [4]:
soy_std_p = round(soy_pess.mean(), 1)
soy_std_b = round(soy_base.mean(), 1)
soy_std_o = round(soy_opt.mean(), 1)

print('=== STANDARD DENSITY — SOYBEANS (real data) ===')
print(f'Pessimistic: {soy_std_p:,} sc/ha  (mean of {len(soy_pess):,} fields)')
print(f'Baseline:    {soy_std_b:,} sc/ha  (mean of {len(soy_base):,} fields)')
print(f'Optimistic:  {soy_std_o:,} sc/ha  (mean of {len(soy_opt):,} fields)')

=== STANDARD DENSITY — SOYBEANS (real data) ===
Pessimistic: 72.6 sc/ha  (mean of 16,515 fields)
Baseline:    84.7 sc/ha  (mean of 16,498 fields)
Optimistic:  97.0 sc/ha  (mean of 16,999 fields)


## 5. Density Alternatives Modelling

The other two densities are modelled with agronomic adjustments applied to the real yields.
The ±20% range is standard in field-trial literature for both soybean and corn density experiments.

| Scenario    | Reduced −20%                          | Increased +20%                     |
|-------------|---------------------------------------|------------------------------------|
| Pessimistic | +5%  (less competition under stress)  | −10% (breaks under stress)         |
| Baseline    | −3%  (lower ceiling)                  | +2%  (slight gain)                 |
| Optimistic  | −8%  (lower productive ceiling)       | +10% (maximum expression)          |

In [5]:
# Reduced Density -20%
soy_red_p = round(soy_std_p * 1.05, 1)
soy_red_b = round(soy_std_b * 0.97, 1)
soy_red_o = round(soy_std_o * 0.92, 1)

# Increased Density +20%
soy_inc_p = round(soy_std_p * 0.90, 1)
soy_inc_b = round(soy_std_b * 1.02, 1)
soy_inc_o = round(soy_std_o * 1.10, 1)

print('=== REDUCED DENSITY -20% ===')
print(f'Pessimistic: {soy_red_p:,} sc/ha  (+5% vs standard)')
print(f'Baseline:    {soy_red_b:,} sc/ha  (-3% vs standard)')
print(f'Optimistic:  {soy_red_o:,} sc/ha  (-8% vs standard)')

print('\n=== INCREASED DENSITY +20% ===')
print(f'Pessimistic: {soy_inc_p:,} sc/ha  (-10% vs standard)')
print(f'Baseline:    {soy_inc_b:,} sc/ha  (+2% vs standard)')
print(f'Optimistic:  {soy_inc_o:,} sc/ha  (+10% vs standard)')

=== REDUCED DENSITY -20% ===
Pessimistic: 76.2 sc/ha  (+5% vs standard)
Baseline:    82.2 sc/ha  (-3% vs standard)
Optimistic:  89.2 sc/ha  (-8% vs standard)

=== INCREASED DENSITY +20% ===
Pessimistic: 65.3 sc/ha  (-10% vs standard)
Baseline:    86.4 sc/ha  (+2% vs standard)
Optimistic:  106.7 sc/ha  (+10% vs standard)


## 6. Full Payoff Matrix — Soybeans

In [6]:
soy_payoff = pd.DataFrame({
    'Alternative' : ['Standard Density', 'Reduced Density -20%', 'Increased Density +20%'],
    'Pessimistic' : [soy_std_p, soy_red_p, soy_inc_p],
    'Baseline'    : [soy_std_b, soy_red_b, soy_inc_b],
    'Optimistic'  : [soy_std_o, soy_red_o, soy_inc_o],
}).set_index('Alternative')

cols = ['Pessimistic', 'Baseline', 'Optimistic']

soy_payoff['Maximax']   = soy_payoff[cols].max(axis=1)
soy_payoff['Maximin']   = soy_payoff[cols].min(axis=1)
soy_payoff['Laplace']   = soy_payoff[cols].mean(axis=1).round(1)

soy_regret              = (soy_payoff[cols].max(axis=0) - soy_payoff[cols]).round(1)
soy_payoff['Max Regret']= soy_regret.max(axis=1)
soy_payoff             = soy_payoff.round(1)

print('=== PAYOFF MATRIX — SOYBEANS (sacas/ha) ===')
print(soy_payoff.to_string())

print('\n=== REGRET MATRIX — SOYBEANS (sacas/ha) ===')
print(soy_regret.to_string())

print('\n=== DECISION SUMMARY — SOYBEANS ===')
print(f'  Maximax  → {soy_payoff["Maximax"].idxmax():<26} {soy_payoff["Maximax"].max():>7,.1f} sc/ha')
print(f'  Maximin  → {soy_payoff["Maximin"].idxmax():<26} {soy_payoff["Maximin"].max():>7,.1f} sc/ha')
print(f'  Laplace  → {soy_payoff["Laplace"].idxmax():<26} {soy_payoff["Laplace"].max():>7,.1f} sc/ha')
print(f'  Savage   → {soy_payoff["Max Regret"].idxmin():<26} {soy_payoff["Max Regret"].min():>7,.1f} sc/ha  (lowest max regret)')

=== PAYOFF MATRIX — SOYBEANS (sacas/ha) ===
                        Pessimistic  Baseline  Optimistic  Maximax  Maximin  Laplace  Max Regret
Alternative                                                                                     
Standard Density               72.6      84.7        97.0     97.0     72.6     84.8         9.7
Reduced Density -20%           76.2      82.2        89.2     89.2     76.2     82.5        17.5
Increased Density +20%         65.3      86.4       106.7    106.7     65.3     86.1        10.9

=== REGRET MATRIX — SOYBEANS (sacas/ha) ===
                        Pessimistic  Baseline  Optimistic
Alternative                                              
Standard Density                3.6       1.7         9.7
Reduced Density -20%            0.0       4.2        17.5
Increased Density +20%         10.9       0.0         0.0

=== DECISION SUMMARY — SOYBEANS ===
  Maximax  → Increased Density +20%       106.7 sc/ha
  Maximin  → Reduced Density -20%          76

---
# PART 2 — CORN

## 7. Filter — Corn Only

In [7]:
corn = df[df['crop_name'] == 'corn'].copy()

print(f'Corn records: {len(corn):,}')
print(f'Seasons: {sorted(corn["concrete_growing_season_name"].unique())}')
print(f'\nYield statistics (average_yield, sacas/ha):')
corn['average_yield'].describe().round(1)

Corn records: 29,075
Seasons: ['2025:corn:BR:1:SUM', '2025:corn:BR:2:SAF', '2026:corn:BR:1:SUM', '2026:corn:BR:2:SAF']

Yield statistics (average_yield, sacas/ha):


count    29075.0
mean        96.7
std         10.6
min         78.3
25%         87.5
50%         96.7
75%        105.9
max        115.0
Name: average_yield, dtype: float64

## 8. Scenario Definition via Terciles

Same methodology as soybeans — scenarios derived from the **real yield distribution split into terciles**:

- **Pessimistic** → bottom tercile (lowest yields = adverse conditions)
- **Baseline** → middle tercile (normal conditions)
- **Optimistic** → top tercile (highest yields = favourable conditions)

In [8]:
t33_corn = corn['average_yield'].quantile(0.33)
t66_corn = corn['average_yield'].quantile(0.66)

print(f'P33 cut (pessimistic / baseline): {t33_corn:.1f} sc/ha')
print(f'P66 cut (baseline / optimistic):  {t66_corn:.1f} sc/ha')

corn_pess = corn[corn['average_yield'] <= t33_corn]['average_yield']
corn_base = corn[(corn['average_yield'] > t33_corn) & (corn['average_yield'] <= t66_corn)]['average_yield']
corn_opt  = corn[corn['average_yield'] > t66_corn]['average_yield']

print(f'\nPessimistic: n={len(corn_pess):,}  range={corn_pess.min():.1f}–{corn_pess.max():.1f} sc/ha')
print(f'Baseline:    n={len(corn_base):,}  range={corn_base.min():.1f}–{corn_base.max():.1f} sc/ha')
print(f'Optimistic:  n={len(corn_opt):,}  range={corn_opt.min():.1f}–{corn_opt.max():.1f} sc/ha')

P33 cut (pessimistic / baseline): 90.4 sc/ha
P66 cut (baseline / optimistic):  102.5 sc/ha

Pessimistic: n=9,602  range=78.3–90.4 sc/ha
Baseline:    n=9,589  range=90.5–102.5 sc/ha
Optimistic:  n=9,884  range=102.5–115.0 sc/ha


## 9. Real Yield Means per Scenario (Standard Density)

The mean yield of each tercile represents **Standard Density** in the Payoff Matrix.

In [9]:
corn_std_p = round(corn_pess.mean(), 1)
corn_std_b = round(corn_base.mean(), 1)
corn_std_o = round(corn_opt.mean(), 1)

print('=== STANDARD DENSITY — CORN (real data) ===')
print(f'Pessimistic: {corn_std_p:,} sc/ha  (mean of {len(corn_pess):,} fields)')
print(f'Baseline:    {corn_std_b:,} sc/ha  (mean of {len(corn_base):,} fields)')
print(f'Optimistic:  {corn_std_o:,} sc/ha  (mean of {len(corn_opt):,} fields)')

=== STANDARD DENSITY — CORN (real data) ===
Pessimistic: 84.4 sc/ha  (mean of 9,602 fields)
Baseline:    96.5 sc/ha  (mean of 9,589 fields)
Optimistic:  108.8 sc/ha  (mean of 9,884 fields)


## 10. Density Alternatives Modelling

Same agronomic adjustment rationale as soybeans:

| Scenario    | Reduced −20%                          | Increased +20%                     |
|-------------|---------------------------------------|------------------------------------|
| Pessimistic | +5%  (less competition under stress)  | −10% (breaks under stress)         |
| Baseline    | −3%  (lower ceiling)                  | +2%  (slight gain)                 |
| Optimistic  | −8%  (lower productive ceiling)       | +10% (maximum expression)          |

In [10]:
# Reduced Density -20%
corn_red_p = round(corn_std_p * 1.05, 1)
corn_red_b = round(corn_std_b * 0.97, 1)
corn_red_o = round(corn_std_o * 0.92, 1)

# Increased Density +20%
corn_inc_p = round(corn_std_p * 0.90, 1)
corn_inc_b = round(corn_std_b * 1.02, 1)
corn_inc_o = round(corn_std_o * 1.10, 1)

print('=== REDUCED DENSITY -20% ===')
print(f'Pessimistic: {corn_red_p:,} sc/ha  (+5% vs standard)')
print(f'Baseline:    {corn_red_b:,} sc/ha  (-3% vs standard)')
print(f'Optimistic:  {corn_red_o:,} sc/ha  (-8% vs standard)')

print('\n=== INCREASED DENSITY +20% ===')
print(f'Pessimistic: {corn_inc_p:,} sc/ha  (-10% vs standard)')
print(f'Baseline:    {corn_inc_b:,} sc/ha  (+2% vs standard)')
print(f'Optimistic:  {corn_inc_o:,} sc/ha  (+10% vs standard)')

=== REDUCED DENSITY -20% ===
Pessimistic: 88.6 sc/ha  (+5% vs standard)
Baseline:    93.6 sc/ha  (-3% vs standard)
Optimistic:  100.1 sc/ha  (-8% vs standard)

=== INCREASED DENSITY +20% ===
Pessimistic: 76.0 sc/ha  (-10% vs standard)
Baseline:    98.4 sc/ha  (+2% vs standard)
Optimistic:  119.7 sc/ha  (+10% vs standard)


## 11. Full Payoff Matrix — Corn

In [11]:
corn_payoff = pd.DataFrame({
    'Alternative' : ['Standard Density', 'Reduced Density -20%', 'Increased Density +20%'],
    'Pessimistic' : [corn_std_p, corn_red_p, corn_inc_p],
    'Baseline'    : [corn_std_b, corn_red_b, corn_inc_b],
    'Optimistic'  : [corn_std_o, corn_red_o, corn_inc_o],
}).set_index('Alternative')

cols = ['Pessimistic', 'Baseline', 'Optimistic']

corn_payoff['Maximax']    = corn_payoff[cols].max(axis=1)
corn_payoff['Maximin']    = corn_payoff[cols].min(axis=1)
corn_payoff['Laplace']    = corn_payoff[cols].mean(axis=1).round(1)

corn_regret               = (corn_payoff[cols].max(axis=0) - corn_payoff[cols]).round(1)
corn_payoff['Max Regret'] = corn_regret.max(axis=1)
corn_payoff              = corn_payoff.round(1)

print('=== PAYOFF MATRIX — CORN (sacas/ha) ===')
print(corn_payoff.to_string())

print('\n=== REGRET MATRIX — CORN (sacas/ha) ===')
print(corn_regret.to_string())

print('\n=== DECISION SUMMARY — CORN ===')
print(f'  Maximax  → {corn_payoff["Maximax"].idxmax():<26} {corn_payoff["Maximax"].max():>7,.1f} sc/ha')
print(f'  Maximin  → {corn_payoff["Maximin"].idxmax():<26} {corn_payoff["Maximin"].max():>7,.1f} sc/ha')
print(f'  Laplace  → {corn_payoff["Laplace"].idxmax():<26} {corn_payoff["Laplace"].max():>7,.1f} sc/ha')
print(f'  Savage   → {corn_payoff["Max Regret"].idxmin():<26} {corn_payoff["Max Regret"].min():>7,.1f} sc/ha  (lowest max regret)')

=== PAYOFF MATRIX — CORN (sacas/ha) ===
                        Pessimistic  Baseline  Optimistic  Maximax  Maximin  Laplace  Max Regret
Alternative                                                                                     
Standard Density               84.4      96.5       108.8    108.8     84.4     96.6        10.9
Reduced Density -20%           88.6      93.6       100.1    100.1     88.6     94.1        19.6
Increased Density +20%         76.0      98.4       119.7    119.7     76.0     98.0        12.6

=== REGRET MATRIX — CORN (sacas/ha) ===
                        Pessimistic  Baseline  Optimistic
Alternative                                              
Standard Density                4.2       1.9        10.9
Reduced Density -20%            0.0       4.8        19.6
Increased Density +20%         12.6       0.0         0.0

=== DECISION SUMMARY — CORN ===
  Maximax  → Increased Density +20%       119.7 sc/ha
  Maximin  → Reduced Density -20%          88.6 sc/ha
  L

---
# PART 3 — SIDE-BY-SIDE COMPARISON

In [12]:
print('=' * 68)
print('  FINAL COMPARISON — SOYBEANS vs CORN')
print('=' * 68)
print(f'  {"Criterion":<10}  {"Soybeans winner":<26}  {"Corn winner"}')
print('-' * 68)
print(f'  {"Maximax":<10}  {soy_payoff["Maximax"].idxmax():<26}  {corn_payoff["Maximax"].idxmax()}')
print(f'  {"Maximin":<10}  {soy_payoff["Maximin"].idxmax():<26}  {corn_payoff["Maximin"].idxmax()}')
print(f'  {"Laplace":<10}  {soy_payoff["Laplace"].idxmax():<26}  {corn_payoff["Laplace"].idxmax()}')
print(f'  {"Savage":<10}  {soy_payoff["Max Regret"].idxmin():<26}  {corn_payoff["Max Regret"].idxmin()}')
print('=' * 68)

  FINAL COMPARISON — SOYBEANS vs CORN
  Criterion   Soybeans winner             Corn winner
--------------------------------------------------------------------
  Maximax     Increased Density +20%      Increased Density +20%
  Maximin     Reduced Density -20%        Reduced Density -20%
  Laplace     Increased Density +20%      Increased Density +20%
  Savage      Standard Density            Standard Density
